# 0.0 Set up

In [36]:
import os
import sys
import importlib
from pathlib import Path

import re
import numpy as np
import pandas as pd

# Change working directory to project directory
project_dir = str(Path.cwd().parent)
if 'modules' not in os.listdir(Path.cwd()):
    os.chdir(project_dir)

In [2]:
from modules import enrollment_preprocessor as ep

# 1.0 Data Preprocessing

## 1.1 Enrollment

### 1.1.1 Module

In [3]:
# Initialize processor
processor = ep.EnrollmentDataProcessor()

In [4]:
# Process data
long_data = processor.process()

# Get summary
summary = processor.get_summary()
print("Data Summary:", summary)

INFO:modules.enrollment_preprocessor:Loaded 60167 records with 100 columns
INFO:modules.enrollment_preprocessor:Transformed to long format: 3489686 records


Data Summary: {'total_records': 3489686, 'total_enrollment': 26916754.0, 'unique_schools': 60167, 'grade_levels': {'G11': 962672, 'G12': 962672, 'K': 120334, 'G1': 120334, 'G2': 120334, 'G3': 120334, 'G4': 120334, 'G5': 120334, 'G6': 120334, 'Elementary': 120334, 'G7': 120334, 'G8': 120334, 'G9': 120334, 'G10': 120334, 'JHS': 120334}, 'academic_tracks': {'ABM': 240668, 'HUMSS': 240668, 'STEM': 240668, 'GAS': 240668, 'PBM': 240668, 'TVL': 240668, 'SPORTS': 240668, 'ARTS & DESIGN': 240668}, 'gender_distribution': {'Male': 1744843, 'Female': 1744843}}


In [5]:
display(long_data)

,Region,Division,District,School ID,School Name,Street Address,Province,Municipality,Legislative District,Barangay,Sector,School Subclassification,School Type,Modified COC,enrollment_count,grade_level,gender,academic_track,student_type
0,Region I,Ilocos Norte,Bacarra I,100001,Apaleng-Libtong ES,"Brgy. 21, Libtong, Bacarra, Ilocos Norte",ILOCOS NORTE,BACARRA,1st District,LIBTONG,Public,DepED Managed,School with no Annexes,Purely ES,4.0,K,Male,None,regular
1,Region I,Ilocos Norte,Bacarra I,100002,Bacarra CES,Santa Rita,ILOCOS NORTE,BACARRA,1st District,SANTA RITA (POB.),Public,DepED Managed,School with no Annexes,Purely ES,26.0,K,Male,None,regular
2,Region I,Ilocos Norte,Bacarra I,100003,Buyon ES,NONE,ILOCOS NORTE,BACARRA,1st District,BUYON,Public,DepED Managed,School with no Annexes,Purely ES,8.0,K,Male,None,regular
3,Region I,Ilocos Norte,Bacarra I,100004,Ganagan Elementary School,"#37 Ganagan,Bacarra, Ilocos Norte",ILOCOS NORTE,BACARRA,1st District,GANAGAN,Public,DepED Managed,School with no Annexes,Purely ES,9.0,K,Male,None,regular
4,Region I,Ilocos Norte,Bacarra I,100005,Macupit ES,Macupit,ILOCOS NORTE,BACARRA,1st District,MACUPIT,Public,DepED Managed,School with no Annexes,Purely ES,5.0,K,Male,None,regular
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3489681,PSO,Italy,Lazio,700034,International Philippine School in Italy,"Via Onofrio Panvinio, 11, 00162 Rome, Italy",NCR SECOND DISTRICT,CITY OF PASIG,Lone District,NaN,PSO,SCHOOL ABROAD,School with no Annexes,ES and JHS,0.0,G12,Female,ARTS & DESIGN,regular
3489682,PSO,Sultanate of Oman,Muscat,700026,Philippine School,4277 Way 2740 AlKhuwair,NCR SECOND DISTRICT,CITY OF PASIG,Lone District,NaN,PSO,SCHOOL ABROAD,School with no Annexes,All Offering,0.0,G12,Female,ARTS & DESIGN,regular
3489683,PSO,State of Kuwait,Jleeb Al-Shuyoukh,700027,The New Kuwait-Philippines International School,Block 1 Jleeb Al-Shuyoukh,NCR SECOND DISTRICT,CITY OF PASIG,Lone District,NaN,PSO,SCHOOL ABROAD,School with no Annexes,All Offering,0.0,G12,Female,ARTS & DESIGN,regular
3489684,PSO,State of Kuwait,Fahaheel,700030,Philippine International English School,Block 11 Street 54 Fahaheel,NCR SECOND DISTRICT,CITY OF PASIG,Lone District,NaN,PSO,SCHOOL ABROAD,School with no Annexes,All Offering,0.0,G12,Female,ARTS & DESIGN,regular


In [6]:
mask = long_data['gender'].isin(['Male','Female'])
df_enr = long_data.loc[mask].copy()
sum_enrollment = df_enr['enrollment_count'].sum()
display(sum_enrollment)

26916754.0

In [7]:
long_data['enrollment_count'].sum()

26916754.0

### 1.1.2 Analog

In [37]:
df_tmp = processor.raw_data.copy()
df_tmp['School ID'] = df_tmp['School ID'].astype('string')

In [38]:
# df_tmp.columns

In [71]:
df_tmp = df_tmp.rename(
    columns={
        'G11ACAD Male':'G11 ACAD Male',
        'G11ACAD Female':'G11 ACAD Female',
        'G12ACAD Male':'G12 ACAD Male',
        'G12ACAD Female':'G12 ACAD Female',
    }
)

cols_profiles = df_tmp.loc[:, "Region":"Modified COC"].columns
cols_data = df_tmp.loc[:, "K Male":"SHS Total"].columns
pattern_exclude = r"total|to|jhs male|jhs female|g11 acad male|g11 acad female|g12 acad male|g12 acad female"
cols_non_total = [col for col in cols_data if not re.search(pattern_exclude, col, flags=re.IGNORECASE)]

# To manually visually check columns
# display(cols_non_total)

In [67]:
pvt_tmp = df_tmp.melt(
    id_vars="School ID",
    value_vars=cols_non_total,
    var_name="column_labels",
    value_name="enrollment_count"
)
pvt_tmp['enrollment_count'] = pd.to_numeric(pvt_tmp['enrollment_count'], errors='coerce')
pvt_tmp = pvt_tmp[pvt_tmp['enrollment_count'].notna()]

In [68]:
pvt_tmp

,School ID,column_labels,enrollment_count
0,100001,K Male,4.0
1,100002,K Male,26.0
2,100003,K Male,8.0
3,100004,K Male,9.0
4,100005,K Male,5.0
...,...,...,...
3489373,406692,G12 ARTS Female,15.0
3489447,488014,G12 ARTS Female,4.0
3489544,407299,G12 ARTS Female,16.0
3489559,305468,G12 ARTS Female,58.0


In [69]:
pvt_tmp['enrollment_count'].sum()

26916754.0